## Folder Structure

Good choice for your scale! Here's a folder structure I'd recommend for a small-business-as-user model:

```
data/receipts/
├── {business_id}/
│   ├── 2026/
│   │   ├── 05/
│   │   │   ├── {receipt_id}.jpg      ← original upload
│   │   │   ├── {receipt_id}.md       ← Datalab markdown output
│   │   │   └── {receipt_name}/         ← extracted images (if any)
│   │   │       ├── page_0_img_0.png
│   │   │       └── ...
│   │   └── 06/
│   └── 2025/
└── {business_id_2}/
    └── ...
```

**Why this layout:**
- **`{business_id}/` at top** → easy per-tenant operations (export, delete, quota check, backup one business)
- **`YYYY/MM/` next** → keeps any one folder small (hundreds of files max), matches how businesses naturally think about receipts ("show me May's receipts"), and makes archiving old years trivial
- **`{receipt_id}` as filename** → unique, no collisions, ties cleanly to the DB row
- **Original + `.md` side by side** → easy to check "does the .md exist?" with a simple file check
- **Subfolder for extracted images** → keeps the main folder clean since Datalab can extract many images per PDF

# Implementation Plan

Implementation Plan:

**1. Schema & Database**
- [x] Define `Business` dataclass (business_id, business_name, created_at)
- [x] Define `User` dataclass (user_id, business_id, user_email, user_name, created_at)
- [x] Update `Receipt` dataclass (business_id FK, uploaded_by_user_id, receipt_name, receipt_mime, file_hash, uploaded_at, processing_status, datalab_request_url, deleted_at)
- [x] Create tables in DB
- [x] Add index on `(business_id, uploaded_at)`
- [x] Seed a single hardcoded business + a few dummy receipts for testing

**2. ID Generation**
- [x] Helper for readable IDs (`biz_xxxxxx`, `rcpt_xxxxxx`)

**3. File Storage Helpers**
- [x] Derive original file path from business_id + uploaded_at + receipt_id + mime
- [x] Derive `.md` path + check existence
- [x] Create folder structure on demand (`data/receipts/{business_id}/YYYY/MM/`)

**4. Upload Flow**
- [x] Compute SHA-256 hash on upload
- [X] Check duplicate (business_id + hash) before saving
- [X] Save original file to derived path
- [X] Insert receipt row with `processing_status = "pending"`
- [X] Call `pdf2md` with our derived path (no changes to `_save_md` needed — it already accepts a path)
- [x] Wrap the call: update status to `"done"` / `"failed"` based on result; 
- [ ] Store `datalab_request_url`
- [ ] Extracted Images (like bar code image from an image or image from a pdf) URL rewriting. Currently, markdown is requesting images from root folder, but these are stored in a sub-folder so need to re-wire these properly to serve on the browser. 

**5. Recently Added Flow**
- [x] Query: most recent 5–10 non-deleted receipts for the hardcoded business
- [x] On reselect: check if `.md` exists on disk → load it; else call Datalab

**6. Web UI Updates**
- [x] Add "Recently Added" section to `/home`
- [-] Show user error when trying to upload a duplicate image.
- [x] Wire reselect → preview + markdown panel
- [x] Upload photo progress bar
- [ ] (Skip login/signup for MVP — single hardcoded business)
- [ ] Try to WYSIWYG view. User should be able to see tables,rows etc as exactly what the image is showing. Mabye convert md2html? 

**7. Soft Delete**
- [ ] Filter `deleted_at IS NULL` in all queries
- [ ] Delete endpoint sets `deleted_at = now()`

In [ ]:
# The simplest way is to stop and restart the server, then re-import:
# import importlib, main
# importlib.reload(main)
# from main import *

In [ ]:
server = JupyUvi(app)
server

In [ ]:
server.stop()